In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
df = pd.read_csv('cleaned_survey_data.csv')

In [4]:
df.head(2)

,EdLevel,WorkExp,DevType,OrgSize,RemoteWork,Country,LanguageHaveWorkedWith,ConvertedCompYearly,lang_JavaScript,lang_SQL,...,lang_Kotlin,lang_Lua,lang_Swift,lang_Visual Basic (.Net),lang_R,lang_Groovy,lang_Assembly,lang_Perl,lang_others,DevType_grouped
0,Some college/university study without earning ...,7.0,"Developer, full-stack",100 to 499 employees,Remote,United States of America,C#;C++;JavaScript;Python;Visual Basic (.Net),87500.0,1,0,...,0,0,0,1,0,0,0,0,0,"Developer, full-stack"
1,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",19.0,"Developer, mobile",500 to 999 employees,Remote,United States of America,Bash/Shell (all shells);C;C#;C++;Java;JavaScri...,205000.0,1,1,...,1,0,1,0,0,0,0,0,0,"Developer, mobile"


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4652 entries, 0 to 4651
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   EdLevel                       4652 non-null   object 
 1   WorkExp                       4652 non-null   float64
 2   DevType                       4652 non-null   object 
 3   OrgSize                       4652 non-null   object 
 4   RemoteWork                    4652 non-null   object 
 5   Country                       4652 non-null   object 
 6   LanguageHaveWorkedWith        4652 non-null   object 
 7   ConvertedCompYearly           4652 non-null   float64
 8   lang_JavaScript               4652 non-null   int64  
 9   lang_SQL                      4652 non-null   int64  
 10  lang_HTML/CSS                 4652 non-null   int64  
 11  lang_Python                   4652 non-null   int64  
 12  lang_Bash/Shell (all shells)  4652 non-null   int64  
 13  lan

In [6]:
df.shape

(4652, 33)

In [17]:
groups = [df[df['DevType_grouped']==dt]['ConvertedCompYearly'] 
          for dt in df['DevType_grouped'].unique()]

f_stats, p_value = stats.f_oneway(*groups)

print(f'f_stats:{f_stats:.3f}\np_value:{p_value:.5f}')

f_stats:28.251
p_value:0.00000


In [18]:
groups = [df[df['RemoteWork']==dt]['ConvertedCompYearly'] 
          for dt in df['RemoteWork'].unique()]

f_stats, p_value = stats.f_oneway(*groups)

print(f'f_stats:{f_stats:.3f}\np_value:{p_value:.5f}')

f_stats:15.393
p_value:0.00000


In [30]:
groups_org = [df[df['OrgSize'] == org]['ConvertedCompYearly'] 
              for org in df['OrgSize'].unique()]

f_stat_org, p_value_org = stats.f_oneway(*groups_org)
print(f'f_stats_org:{f_stat_org:.3f}\np_value_org:{p_value_org:.5f}')

f_stats:30.394
p_value:0.00000


In [32]:
groups_edu = [df[df['EdLevel'] == edu]['ConvertedCompYearly'] 
              for edu in df['EdLevel'].unique()]

f_stat_edu, p_value_edu = stats.f_oneway(*groups_edu)
print(f'f_stats_edu:{f_stat_edu:.3f}\np_value_edu:{p_value_edu:.5f}')

f_stats_edu:12.654
p_value_edu:0.00000


Based on this data, what organization size and role someone has appears to be is a stronger salary differentiator than their education level, remote-work arrangement

Formal education level showed the weakest relationship with salary among the categorical predictors tested - while still statistically significant, its effect (F=12.654) was meaningfully smaller than organization size, role, or remote-work arrangement.

In [24]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(
    endog=df['ConvertedCompYearly'],
    groups=df['DevType_grouped'],
    alpha=0.05
)
print(tukey_result)

                                             Multiple Comparison of Means - Tukey HSD, FWER=0.05                                              
                    group1                                        group2                      meandiff  p-adj     lower        upper    reject
----------------------------------------------------------------------------------------------------------------------------------------------
                               AI/ML engineer              Architect, software or solutions   10592.289    0.9  -26963.5891   48148.167  False
                               AI/ML engineer                                 Data engineer -23476.6671 0.8545  -66797.2745  19843.9403  False
                               AI/ML engineer                      Data or business analyst -86378.1749  0.001 -142195.4556 -30560.8941   True
                               AI/ML engineer                                Data scientist -40400.8479  0.149   -86031.029   5229.3333  False

Post-hoc testing (Tukey HSD) shows the ANOVA's significant result is driven primarily by a few standout groups - 
* Engineering managers and Architects earn significantly more than most individual - contributor roles. 
* Data/business analysts earn significantly less than nearly all other roles 
* While most remaining roles are not statistically distinguishable from one another.